## 1. Dataset

repository_url: ```https://www.kaggle.com/datasets/sujalsuthar/amazon-delivery-dataset```

Uncover insights into factors influencing delivery efficiency, identify areas for optimization, and explore the impact of various variables on the overall customer experience

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
from importlib import reload

import seaborn as sns
from collections import defaultdict
from tqdm import tqdm
import inspect


sys.path.append('/Users/cheungjustin/PycharmProjects/CIBer')
import util as put
import stats as pstats
import analytics as pan
import plot as pplot

In [ ]:
df = pd.read_csv('../Raw/amazon_delivery.csv', index_col=0)
df['Order_Date'] = pd.to_datetime(df['Order_Date'])
df['Order_Time'] = pd.to_timedelta(df['Order_Time'], errors='coerce') / np.timedelta64(1, 'm')
df['Pickup_Time'] = pd.to_timedelta(df['Pickup_Time'], errors='coerce') / np.timedelta64(1, 'm')

thres = 4.5
df['Dissatisfied'] = (df['Agent_Rating'] < thres)#.astype(int)
df['Delivery_Time'] = df['Delivery_Time'].astype(float)
df = df.drop(columns='Agent_Rating')

display(df)
print(df.dtypes)

## 2. EDA Wrapper

In [ ]:
random_state = 4012
target = 'Dissatisfied'
dta = pan.DataAnalyzer(df, 
                       target=target, cont_thres=30, random_state=random_state)

In [ ]:
dta.data_quality_check(check_zero=True, miss_thres=0.);

In [ ]:
dta.dist_plot();

In [ ]:
dta.corr_heatmap(corr_method='dcorr', cluster=True);
dta.corr_heatmap(corr_method='dcorr', partial=True, cluster=True, linkage_method='single');

### 2.1 Update Features

In [ ]:
df2 = df.copy()
missing_store = (df2[['Store_Latitude', 'Store_Longitude']].abs() < 1e-6).all(axis=1)
df2.loc[missing_store, ['Store_Latitude', 'Store_Longitude']] = np.nan

def calc_row_dist(row):
    store = (row['Store_Latitude'], row['Store_Longitude'])
    drop = (row['Drop_Latitude'], row['Drop_Longitude'])
    if (np.isfinite([*store, *drop])).all():
        return geopy.distance.geodesic(store, drop).km
    else:
        return np.nan
    
df2['Drop_Distance'] = df2.apply(calc_row_dist, axis=1)

In [ ]:
wait_time = (df2['Pickup_Time'] - df2['Order_Time'])
df2['Wait_Time'] = wait_time[wait_time >= 0]

In [ ]:
df2 = df2.drop(columns=['Drop_Latitude', 'Drop_Longitude', 'Pickup_Time'])
df2

In [ ]:
dta = pan.DataAnalyzer(df2, 
                       target=target, cont_thres=30, random_state=random_state)

In [ ]:
dta.data_quality_check(miss_thres=0.);

In [ ]:
dta.dist_plot();

In [ ]:
dta.corr_heatmap(corr_method='dcorr', cluster=True);
dta.corr_heatmap(corr_method='dcorr', partial=True, cluster=True, linkage_method='single');

In [ ]:
dta.pairplot(y_vars='Delivery_Time')

In [ ]:
dta.dist_plot_by_categ('Traffic', kind='hist');

In [ ]:
dta.df.to_csv('../Processed/Amazon.csv')
print(dta.cont_cols)
dta.df

In [ ]:
dta.df.dtypes